# 📝 Pipeline: NLP con LSTM — Análisis de Sentimientos

**Propósito:** Clasificar reseñas de texto como positivas o negativas usando
una LSTM bidireccional con embeddings aprendidos desde cero.

---
## 📋 Flujo de trabajo

1. Cargar dataset IMDB (keras.datasets)
2. Preprocesar: tokenizar + padding con `preprocesar_texto()`
3. Crear LSTM con `crear_lstm_texto()`
4. Entrenar con `compilar_y_entrenar()`
5. Evaluar con `evaluar_clasificacion()`
6. Probar con reseñas personalizadas

In [ ]:
# MACHOTE — Setup universal
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

VOCAB_SIZE = 10000
MAX_LEN = 200

In [ ]:
# MACHOTE — Cargar IMDB Dataset

print("Cargando dataset IMDB...")
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

print(f"Train: {len(X_train_raw)} reseñas")
print(f"Test:  {len(X_test_raw)} reseñas")
print(f"Clases: 0 = Negativa, 1 = Positiva")
print(f"\nDistribución train: {np.bincount(y_train)}")
print(f"Distribución test:  {np.bincount(y_test)}")

In [ ]:
# MACHOTE — Decodificar una reseña para inspección visual

# Obtener el mapping índice → palabra del dataset IMDB
word_index = keras.datasets.imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0] = '<PAD>'
index_to_word[1] = '<START>'
index_to_word[2] = '<OOV>'


def decodificar_resena(seq):
    return ' '.join(index_to_word.get(i, '?') for i in seq)


print("🔹 Ejemplo de reseña POSITIVA:")
print(decodificar_resena(X_train_raw[y_train == 1][0])[:500])
print("\n" + "="*60)
print("🔸 Ejemplo de reseña NEGATIVA:")
print(decodificar_resena(X_train_raw[y_train == 0][0])[:500])

In [ ]:
# MACHOTE — Preprocesar: padding de secuencias

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test  = pad_sequences(X_test_raw, maxlen=MAX_LEN, padding='pre', truncating='pre')

# Separar validation del train
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"\nCada reseña se convirtió en secuencia de {MAX_LEN} índices.")

In [ ]:
# MACHOTE — Crear LSTM Bidireccional

lstm = crear_lstm_texto(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embedding_dim=64,
    num_clases=2
)
lstm.summary()

In [ ]:
# MACHOTE — Entrenar LSTM (puede tomar varios minutos en CPU)

hist = compilar_y_entrenar(
    lstm,
    X_train, y_train,
    X_val, y_val,
    num_clases=2,
    epochs=10,
    batch_size=64,
    early_stop_paciencia=3
)

graficar_historia(hist, 'LSTM — IMDB Sentiment')

In [ ]:
# MACHOTE — Evaluar en test

preds = evaluar_clasificacion(lstm, X_test, y_test, nombres_clases=['Negativo', 'Positivo'])

In [ ]:
# MACHOTE — Probar con reseñas personalizadas

from tensorflow.keras.preprocessing.text import Tokenizer


def predecir_sentimiento(texto, modelo, tokenizador, max_len=200):
    """Tokeniza un texto y devuelve predicción."""
    seq = tokenizador.texts_to_sequences([texto])
    pad = pad_sequences(seq, maxlen=max_len, padding='pre', truncating='pre')
    prob = modelo.predict(pad, verbose=0)[0][0]
    return prob


# Crear tokenizador a partir del dataset original para mapear palabras
tokenizador_imdb = Tokenizer(num_words=VOCAB_SIZE)
# Reconstruir textos para fit
textos_train = [decodificar_resena(r) for r in X_train_raw]
tokenizador_imdb.fit_on_texts(textos_train)

# Frases de prueba
frases = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "A complete waste of time. Terrible acting and boring plot.",
    "It was okay, not great but not terrible either.",
    "The cinematography was stunning but the story was weak.",
]

print("📊 Predicciones en frases personalizadas:\n")
for frase in frases:
    prob = predecir_sentimiento(frase, lstm, tokenizador_imdb, MAX_LEN)
    sentimiento = "POSITIVO 😊" if prob > 0.5 else "NEGATIVO 😞"
    print(f"  [{sentimiento}] ({prob:.1%}) → \"{frase}\"")

---
## 🔄 Comparación: LSTM vs GRU

| Aspecto | LSTM | GRU |
|---------|------|-----|
| Compuertas | 3 (input, forget, output) | 2 (reset, update) |
| Parámetros | Más (~4x tamaño oculto) | Menos (~3x tamaño oculto) |
| Velocidad | Más lento | ~20-30% más rápido |
| Memoria larga | ✅ Excelente | ✅ Muy buena |
| Datos pequeños | Puede sobreajustar | Generaliza mejor |

> **Regla práctica:** GRU para datasets medianos/pequeños, LSTM para grandes.
> Ambos superan a RNN simple (vanishing gradient).

### 💡 Tips para NLP

1. **Preprocessing:** eliminar HTML, URLs, reducir a minúsculas, stemming/lemmatization
2. **Vocab size:** 10,000–50,000 es típico; más no siempre mejora
3. **Embeddings pre-entrenados:** GloVe, Word2Vec, FastText mejoran con datos pequeños
4. **Bidirectional LSTM:** Lee el texto en ambas direcciones (default en `crear_lstm_texto()`)
5. **Attention:** Mecanismo de atención mejora interpretabilidad

---
*Pipeline creado para el Diplomado en Redes Neuronales — Módulo 4* 🧠

In [ ]:
# TODO: Experimenta con estas variaciones
# 1. Cambia embedding_dim=128 — ¿mejora el accuracy?
# 2. Cambia VOCAB_SIZE=20000
# 3. Prueba con MAX_LEN=100 vs MAX_LEN=500
# 4. ¿Qué pasa si usas solo LSTM (no Bidirectional)?
# 5. Implementa una GRU en lugar de LSTM
#    (en la librería no hay crear_gru, pero puedes definirla tú)
# Pregunta: ¿Por qué las reseñas cortas se clasifican mejor o peor?